# Spotify Listening Time — Pipeline MLOps Complet

Prédiction du temps d'écoute (`listening_time`) via régression linéaire avec GridSearch.

**Pipeline :** EDA → Preprocessing → Modélisation → Comparaison des modèles → Évaluation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('Librairies chargées')

---
## 1. Chargement et exploration des données

In [ ]:
df = pd.read_csv('../data/raw/spotify_churn_dataset.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribution target
axes[0].hist(df['listening_time'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution — listening_time', fontsize=13)
axes[0].set_xlabel('listening_time')

# Valeurs manquantes
missing = df.isnull().sum()
axes[1].bar(missing.index, missing.values, color='salmon')
axes[1].set_title('Valeurs manquantes par colonne', fontsize=13)
axes[1].tick_params(axis='x', rotation=45)

# Types de subscription
df['subscription_type'].value_counts().plot(kind='bar', ax=axes[2], color='mediumseagreen', edgecolor='white')
axes[2].set_title('Répartition subscription_type', fontsize=13)
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 2. Corrélations et relations avec la target

In [ ]:
numerical_cols = ['age', 'songs_played_per_day', 'skip_rate', 'ads_listened_per_week', 'offline_listening', 'listening_time']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap
corr = df[numerical_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[0], square=True)
axes[0].set_title('Matrice de corrélation', fontsize=13)

# Corrélation avec target
target_corr = corr['listening_time'].drop('listening_time').sort_values()
colors = ['salmon' if x < 0 else 'steelblue' for x in target_corr]
target_corr.plot(kind='barh', ax=axes[1], color=colors, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Corrélation avec listening_time', fontsize=13)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.groupby('subscription_type')['listening_time'].mean().sort_values().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('listening_time moyen par subscription_type', fontsize=13)
axes[0].tick_params(axis='x', rotation=30)

df.groupby('device_type')['listening_time'].mean().sort_values().plot(
    kind='bar', ax=axes[1], color='mediumseagreen', edgecolor='white')
axes[1].set_title('listening_time moyen par device_type', fontsize=13)
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

---
## 3. Preprocessing

In [ ]:
TARGET = 'listening_time'
CATEGORICAL = ['gender', 'country', 'subscription_type', 'device_type']
NUMERICAL = ['age', 'songs_played_per_day', 'skip_rate', 'ads_listened_per_week', 'offline_listening']

df_clean = df.drop(columns=['user_id', 'is_churned'])
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUMERICAL),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL),
])

X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)

print(f'Train: {X_train_p.shape} | Test: {X_test_p.shape}')

---
## 4. GridSearch — Comparaison des modèles

In [ ]:
MODELS_GRID = {
    'LinearRegression': {
        'model': LinearRegression(),
        'params': {'fit_intercept': [True, False]}
    },
    'Ridge': {
        'model': Ridge(),
        'params': {'alpha': [0.1, 1.0, 10.0, 100.0], 'fit_intercept': [True, False]}
    },
    'Lasso': {
        'model': Lasso(max_iter=5000),
        'params': {'alpha': [0.01, 0.1, 1.0, 10.0], 'fit_intercept': [True, False]}
    },
}

results = []
best_models = {}

for name, config in MODELS_GRID.items():
    grid = GridSearchCV(config['model'], config['params'], cv=5, scoring='r2', n_jobs=-1)
    grid.fit(X_train_p, y_train)
    best = grid.best_estimator_
    best_models[name] = best
    y_pred = best.predict(X_test_p)
    results.append({
        'Model': name,
        'Best params': grid.best_params_,
        'CV R²': round(grid.best_score_, 4),
        'Test R²': round(r2_score(y_test, y_pred), 4),
        'Test RMSE': round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
        'Test MAE': round(mean_absolute_error(y_test, y_pred), 4),
    })
    print(f'{name} — CV R²: {grid.best_score_:.4f} | Test R²: {r2_score(y_test, y_pred):.4f}')

results_df = pd.DataFrame(results)
results_df

## 5. Visualisation comparative des modèles

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ['CV R²', 'Test R²', 'Test RMSE']
colors = ['steelblue', 'mediumseagreen', 'salmon']

for ax, metric, color in zip(axes, metrics, colors):
    bars = ax.bar(results_df['Model'], results_df[metric], color=color, edgecolor='white')
    ax.set_title(metric, fontsize=13)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Comparaison des modèles (GridSearch)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Évaluation du meilleur modèle

In [ ]:
best_name = results_df.loc[results_df['Test R²'].idxmax(), 'Model']
best_model = best_models[best_name]
y_pred_best = best_model.predict(X_test_p)
residuals = y_test.values - y_pred_best

print(f'Meilleur modèle : {best_name}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred_best, alpha=0.4, color='steelblue', edgecolors='white', s=20)
lims = [min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())]
axes[0].plot(lims, lims, 'r--', lw=2)
axes[0].set_xlabel('Valeurs réelles')
axes[0].set_ylabel('Valeurs prédites')
axes[0].set_title(f'{best_name} — Actual vs Predicted', fontsize=12)

# Résidus
axes[1].scatter(y_pred_best, residuals, alpha=0.4, color='mediumseagreen', edgecolors='white', s=20)
axes[1].axhline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Valeurs prédites')
axes[1].set_ylabel('Résidus')
axes[1].set_title('Résidus', fontsize=12)

# Distribution résidus
axes[2].hist(residuals, bins=40, color='salmon', edgecolor='white')
axes[2].axvline(0, color='black', linestyle='--', lw=2)
axes[2].set_title('Distribution des résidus', fontsize=12)
axes[2].set_xlabel('Résidu')

plt.suptitle(f'Évaluation — {best_name}', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Importance des features (coefficients)

In [ ]:
num_features = NUMERICAL
cat_features = preprocessor.named_transformers_['cat'].get_feature_names_out(CATEGORICAL).tolist()
all_features = num_features + cat_features

coefs = pd.Series(best_model.coef_, index=all_features)
top_coefs = coefs.abs().nlargest(15).index

plt.figure(figsize=(10, 6))
colors = ['steelblue' if v > 0 else 'salmon' for v in coefs[top_coefs]]
coefs[top_coefs].sort_values().plot(kind='barh', color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.title(f'Top 15 coefficients — {best_name}', fontsize=13)
plt.tight_layout()
plt.show()